In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib
import re

In [2]:
runners = pd.read_excel("../../data/cleaned/results/runners.xlsx")

In [9]:
medical = pd.read_excel("../../data/cleaned/acceptances/medical.xlsx")

In [ ]:
# 1. Get unique names and clean them to find potential duplicates
unique_names = medical.horse_name.dropna().unique()

# 2. Look for close matches
similar_pairs = []
for i, name1 in enumerate(unique_names):
    # Get close matches from the rest of the list
    matches = difflib.get_close_matches(name1, unique_names[i+1:], n=3, cutoff=0.8)
    for match in matches:
        similar_pairs.append((name1, match))

# 3. View the results clearly
df_typos = pd.DataFrame(similar_pairs, columns=['Name A', 'Name B'])
df_typos

,Name A,Name B
0,SPARKLING THEA,SPARKLING DEW
1,RARE SILVER,STAR SILVER
2,ADELE,ADELINE
3,LUCAS,LUCA
4,GOLDEN TOWER,GOLDEN ERA
5,DIVINE SPARK,DIVINE STAR
6,GOLDEN EVE,GOLDEN ERA
7,ZANARA,ANAIRA
8,MARIANELLA,MARIELLA
9,WESTERN STYLE,WESTERN STAR


In [4]:
unique_names = medical.horse_name.dropna().unique()

# Create a DataFrame of unique names
df_names = pd.DataFrame({'original_name': unique_names})

# Create a 'clean' key: uppercase, no spaces, no punctuation
df_names['clean_key'] = df_names['original_name'].apply(
    lambda x: re.sub(r'[^A-Z0-9]', '', str(x).upper())
)

# Find clean keys that appear more than once (meaning they have variations)
duplicates = df_names[df_names.duplicated(subset=['clean_key'], keep=False)]

# Sort them so variations sit next to each other
duplicates.sort_values(by='clean_key')

,original_name,clean_key


In [13]:
# =========================
# 1. NORMALIZE
# =========================
def normalize(df):
    df = df.copy()
    df['meet_date'] = pd.to_datetime(df['meet_date'])
    df['horse_name'] = df['horse_name'].astype(str).str.strip().str.upper()
    return df

runners = normalize(runners)
medical = normalize(medical)

# =========================
# 2. BUILD RUNNER MAP
# =========================
runner_map = runners[['meet_date', 'horse_name', 'race_no']].drop_duplicates()

# group for partial matching
runner_grouped = runner_map.groupby('meet_date')

# =========================
# 3. EXACT MERGE
# =========================
merged = medical.merge(
    runner_map,
    on=['meet_date', 'horse_name'],
    how='left',
    indicator=True,
    suffixes=('_old', '_true')
)

exact_matched = merged[merged['_merge'] == 'both'].copy()
unmatched = merged[merged['_merge'] == 'left_only'].copy()

# =========================
# 4. PARTIAL MATCH (PREFIX ONLY)
# =========================
resolved_rows = []
still_unmatched = []

for _, row in unmatched.iterrows():
    date = row['meet_date']
    name = row['horse_name']
    
    if date not in runner_grouped.groups:
        still_unmatched.append(row)
        continue
    
    candidates = runner_grouped.get_group(date)
    
    matches = candidates[
        candidates['horse_name'].apply(lambda x: x.startswith(name))
    ]
    
    if len(matches) >= 1:
        # pick best match (longest name)
        best = matches.loc[matches['horse_name'].str.len().idxmax()]
        
        row['race_no'] = best['race_no']
        row['horse_name'] = best['horse_name']   # FIX name
        
        row['_merge'] = 'partial'
        resolved_rows.append(row)
    else:
        still_unmatched.append(row)

partial_matched = pd.DataFrame(resolved_rows)
edge_cases = pd.DataFrame(still_unmatched)

# =========================
# 5. CLEAN EXACT MATCH
# =========================
exact_matched['race_no'] = exact_matched['race_no_true']
# names already correct in exact match

exact_matched = exact_matched.drop(
    columns=['race_no_old', 'race_no_true', '_merge']
)

# =========================
# 6. FINAL COMBINE
# =========================
clean_medical = pd.concat([exact_matched, partial_matched], ignore_index=True)

# =========================
# 7. ADD VALIDITY FLAG
# =========================
clean_medical['is_valid_runner'] = True

# handle non-runners (keep them)
if len(edge_cases) > 0:
    edge_cases['race_no'] = edge_cases['race_no_old']   # keep original
    edge_cases['is_valid_runner'] = False
    
    edge_cases = edge_cases.drop(columns=['race_no_true', '_merge'])
    
    clean_medical = pd.concat([clean_medical, edge_cases], ignore_index=True)

# =========================
# 8. SORT + FORMAT
# =========================
clean_medical = clean_medical.sort_values(
    by=['meet_date', 'race_no']
).reset_index(drop=True)

clean_medical['meet_date'] = clean_medical['meet_date'].dt.strftime('%Y-%m-%d')
clean_medical['date'] = clean_medical['date'].dt.strftime('%Y-%m-%d')

# =========================
# 9. DEBUG OUTPUT
# =========================
print("Valid runners:", clean_medical['is_valid_runner'].sum())
print("Non-runners:", (~clean_medical['is_valid_runner']).sum())

# =========================
# 10. SAVE
# =========================
clean_medical.to_excel("../../data/cleaned/acceptances_cleaned/medical.xlsx", index=False)

Valid runners: 13385
Non-runners: 243
